In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
from scipy.stats import pearsonr

In [ ]:
def binding_curve(conc, Kd, low, high):
    return (high - low)*(conc/(conc + Kd)) + low

In [ ]:
data = pd.read_csv('../figS16_EC50-curve-fitting/ciBoNTA_dilution_ELISA_data.csv')
data

In [ ]:
fit_params = []

colors = plt.get_cmap('tab10')
for i in range(0,len(data.columns)-1,10):
    # plt.figure()
    for ii,col in enumerate(data.columns[i+1:min(i+11,len(data.columns))]):
        # plt.plot(data['concs'],data[col],'o',color=colors(ii))

        params,cov = curve_fit(binding_curve, data['concs'], data[col], p0=[1,0,2])
        # print(col,params)

        # x_fit = np.logspace(-3,2.2,1000)
        # y_fit = binding_curve(x_fit,*params)
        # plt.plot(x_fit,y_fit,'-',color=colors(ii))

        fit_params.append([col]+list(params)+list(np.sqrt(np.diag(cov))))
        
    # plt.semilogx()

fit_params = pd.DataFrame(fit_params,columns=['name','Kd','low','high','Kd_err','low_err','high_err'])

In [ ]:
vhh = pd.read_csv('../large_files/JYH3_factor1_gfold_vhh.csv')
fam = pd.read_csv('../large_files/JYH3_factor1_gfold_fam.csv')

cols = ['VHH','enrich_ciBoNTA-JDA','gfold01_ciBoNTA-JDA','clone_id_1_5_3','1s_ciBoNTA-JDA','1l_ciBoNTA-JDA']

seqs = pd.read_csv('../figS16_EC50-curve-fitting/selected_sequences.csv').drop_duplicates('VHH name')
seqs = seqs.merge(vhh[cols],left_on='sequence',right_on='VHH',how='left')
seqs = seqs.merge(fam[cols[1:]],on='clone_id_1_5_3',how='left',suffixes=('_vhh','_fam'))
del seqs['sequence']
seqs.head()

In [ ]:
for kind in ['vhh','fam']:
    seqs[f'tot_ciBoNTA-JDA_{kind}'] = seqs[[f'1s_ciBoNTA-JDA_{kind}',f'1l_ciBoNTA-JDA_{kind}']].sum(axis=1)

In [ ]:
fit_params = fit_params.merge(seqs,left_on='name',right_on='VHH name',how='left')
fit_params.head()

In [ ]:
print(len(fit_params))
fit_params = fit_params[fit_params['clone_id_1_5_3'].notna()].reset_index()
print(len(fit_params))

In [ ]:
for name in fit_params['name']:
    print(name)

In [ ]:
for method in ['gfold01','enrich','tot']:
    for level in ['vhh','fam']:
        result = pearsonr(x=fit_params[f'{method}_ciBoNTA-JDA_{level}'],y=np.log10(fit_params['Kd']))
        print(method, level, result.statistic, result.pvalue)

In [ ]:
letters = 'AB'
labels = {
    'tot':'Total Reads',
    'gfold01':'GFold'
}
plt.figure(figsize=[8.7,4.3])
for k,kind in enumerate(['tot','gfold01']):
    x = fit_params[f'{kind}_ciBoNTA-JDA_fam']
    if kind == 'gfold01':
        result = pearsonr(x=x,y=np.log10(fit_params['Kd']))
    else:
        result = pearsonr(x=np.log10(x),y=np.log10(fit_params['Kd']))

    plt.subplot(1,2,k+1)
    ax = sns.scatterplot(x=x,y=fit_params['Kd'])
    if kind == 'gfold01':
        plt.semilogy()
    else:
        plt.loglog()

    if k==0:
        plt.ylabel('EC$_{50}$ (nM)',fontsize=16)
    else:
        plt.ylabel('')
    plt.xlabel(labels[kind],fontsize=16)

    plt.text(s=f'R = {round(result.statistic,2)}\np = {result.pvalue:.2e}',x=0.63,y=0.88,fontsize=12,transform=ax.transAxes)
    plt.text(s=letters[k],x=-0.15,y=1.05,fontsize=32,transform=ax.transAxes)
plt.savefig('fig5.png',dpi=300,bbox_inches='tight')